# 1. Imports & Configuration setup

In [6]:
# ================================== Imports ====================================
import sys
import numpy as np
import cv2
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from scipy.spatial import cKDTree # For efficient nearest neighbor search

# Nuscenes SDK
from nuscenes.nuscenes import NuScenes
from pyquaternion import Quaternion

# Set DPI (dots per inch) for better resolution in plots
matplotlib.rcParams['figure.dpi'] = 120

# Enable inline plotting in Jupyter notebooks
%matplotlib inline

print('Imports OK')

In [4]:
# ================================== Configuration ==================================

# Set scene and frame
NUSCENES_SCENE_IDX = 10
FRAME_IDX          = 1580

# Set paths 
DATA_ROOT          = '/media/lleba/ECP_Nuscenes_01/output2/ecp2nuscenes'

SAM3_MASK_DIR      = Path('/media/lleba/ECP_Nuscenes_01/SAM3_Visualizations/Strassbourg')
SAM3_SCENE_PREFIX  = f'scene_{NUSCENES_SCENE_IDX}_cam_0_sam3_outputs__'

SAM3D_REPO         = Path('/home/lleba/Thesis/Development/SAM3D/sam-3d-objects')
CKPT_CONFIG        = SAM3D_REPO / 'checkpoints' / 'hf' / 'pipeline.yaml'

# Set object class and select objects
# possible object classes: 'car', 'motorcycle', 'bus', 'construction_vehicle', 'trailer', 'truck', 'bicycle and cyclist'
OBJ_CLASS = 'car'
OBJ_IDXS  = [0, 1, 2]   #   use a list  [0, 1, 2]  or 'all' for every detection

# SAM3 mask filtering thresholds
MIN_PROB     = 0.6      # Minimum SAM3 mask probability to consider for depth estimation
MIN_MASK_PX  = 100      # Minimum SAM3 mask size in pixels to attempt depth estimation
MIN_LIDAR_HITS = 3      # Minimum in-mask LiDAR hits to trust the depth anchor

# Per-class maximum plausible depth span in meters (depth-window filter)
CLASS_DEPTH_SPAN = {
    'car':                  3.0,
    'motorcycle':           2.5,
    'bus':                 13.0,
    'construction_vehicle': 8.0,
    'trailer':             13.0,
    'truck':                8.0,
    'bicycle and cyclist':  2.0,
}
MAX_DEPTH_SPAN = CLASS_DEPTH_SPAN[OBJ_CLASS]

# Robust MAD (median absolute deviation) outlier filter (applied after depth-window filter, before local affine fit)
MAD_SIGMA_THRESH = 3.0   # reject points whose |residual| > MAD_SIGMA_THRESH * sigma_MAD
MIN_MOGE_SPAN   = 0.05   # min MoGe depth range (normalised) to attempt fit
MIN_LIDAR_SPAN  = 0.30   # min LiDAR depth span (m) to attempt fit

print(f'Config: scene={NUSCENES_SCENE_IDX}  frame={FRAME_IDX}  class={OBJ_CLASS}  obj_idxs={OBJ_IDXS}')


# 2. Load SAM 3D Model

In [5]:
# ================================== Load SAM3D-Objects model ==================================
import torch
sys.path.insert(0, str(SAM3D_REPO / 'notebook'))
from inference import Inference

inference_model = Inference(str(CKPT_CONFIG), compile=False)
print('SAM3D-Objects model ready.')

# 3. Load scene data — calibration, camera frame, nearest LiDAR sweep

Loads all calibration matrices (camera intrinsics, camera -- ego, ego -- LiDAR),
reads the camera image, and finds the temporally closest LiDAR sweep.

In [7]:
print('Loading nuScenes...')
nusc  = NuScenes(version='v1.0-trainval', dataroot=DATA_ROOT, verbose=False)
scene = nusc.scene[NUSCENES_SCENE_IDX]
print(f'  Scene: {scene["name"]}')

# Walk CAM_FRONT frames
cam_token = nusc.get('sample', scene['first_sample_token'])['data']['CAM_FRONT']
frame_records = []
while cam_token:
    rec = nusc.get('sample_data', cam_token)
    frame_records.append(rec)
    cam_token = rec['next']
print(f'  {len(frame_records)} CAM_FRONT frames')

rec_cam  = frame_records[FRAME_IDX]
img_path = nusc.get_sample_data_path(rec_cam['token'])

cal_cam   = nusc.get('calibrated_sensor', rec_cam['calibrated_sensor_token'])
K_np      = np.array(cal_cam['camera_intrinsic'], dtype=np.float64)   # (3,3)
fx, fy    = K_np[0,0], K_np[1,1]
cx, cy    = K_np[0,2], K_np[1,2]
R_cam2ego = Quaternion(cal_cam['rotation']).rotation_matrix            # (3,3)
t_cam2ego = np.array(cal_cam['translation'], dtype=np.float64)         # (3,)
R_ego2cam = R_cam2ego.T
t_ego2cam = -R_ego2cam @ t_cam2ego

print(f'  ECP intrinsics: fx={fx:.1f}  fy={fy:.1f}  cx={cx:.1f}  cy={cy:.1f}')
print(f'  Image: {img_path}')

In [8]:
# Find LiDAR sweep nearest in time to this camera frame
lt = nusc.get('sample', scene['first_sample_token'])['data']['LIDAR_TOP']
lidar_records = []
while lt:
    lr = nusc.get('sample_data', lt)
    lidar_records.append(lr)
    lt = lr['next']

lidar_ts  = np.array([lr['timestamp'] for lr in lidar_records], dtype=np.int64)
best_idx  = int(np.argmin(np.abs(lidar_ts - rec_cam['timestamp'])))
rec_lidar = lidar_records[best_idx]
dt_ms     = abs(rec_lidar['timestamp'] - rec_cam['timestamp']) / 1000.0

lidar_path = nusc.get_sample_data_path(rec_lidar['token'])
cal_lidar  = nusc.get('calibrated_sensor', rec_lidar['calibrated_sensor_token'])
R_l2e      = Quaternion(cal_lidar['rotation']).rotation_matrix     # (3,3) lidar→ego
t_l2e      = np.array(cal_lidar['translation'], dtype=np.float64)  # (3,)

print(f'  LiDAR sweep: {lidar_path}')
print(f'  Time offset to camera: {dt_ms:.1f} ms')

# 4. Transform LiDAR to camera space and project to image

Transforms the full LiDAR sweep through ego space into camera space,
then applies pinhole projection to get pixel coordinates for each point.


In [13]:
# Load LiDAR point cloud (.bin = float32, columns: x, y, z, intensity, ring)
pc_raw    = np.fromfile(lidar_path, dtype=np.float32).reshape(-1, 5)
pts_lidar = pc_raw[:, :3].T   # (3, N) in LiDAR frame
print(f'  Loaded {pts_lidar.shape[1]:,} LiDAR points')

# lidar → ego → camera (R3/OpenCV: X right, Y down, Z forward)
pts_ego = R_l2e @ pts_lidar + t_l2e[:, np.newaxis]
pts_cam = R_ego2cam @ pts_ego + t_ego2cam[:, np.newaxis]

# Keep only points in front of camera
in_front = pts_cam[2] > 0.5
pts_cam  = pts_cam[:, in_front]
print(f'  {pts_cam.shape[1]:,} points in front of camera')

# Pinhole projection
Z_all = pts_cam[2]
u_all = fx * pts_cam[0] / Z_all + cx
v_all = fy * pts_cam[1] / Z_all + cy

img_bgr = cv2.imread(img_path)
H, W    = img_bgr.shape[:2]
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

# Keep only points that project within image bounds
in_img      = (u_all >= 0) & (u_all < W) & (v_all >= 0) & (v_all < H)
u_vis       = u_all[in_img]
v_vis       = v_all[in_img]
Z_vis       = Z_all[in_img]
pts_cam_vis = pts_cam[:, in_img]
print(f'  {pts_cam_vis.shape[1]:,} points project into image')
xyz_lid = pts_lidar.T.astype(np.float64)  # (N,3) LiDAR frame

# convert LiDAR points to pixel coordinates (for efficient in-mask point retrieval)
u_int     = np.round(u_vis).astype(int)
v_int     = np.round(v_vis).astype(int)
in_bounds = (u_int >= 0) & (u_int < W) & (v_int >= 0) & (v_int < H)
u_int     = u_int[in_bounds]
v_int     = v_int[in_bounds]
Z_v       = Z_vis[in_bounds]
u_f_v     = u_vis[in_bounds]
v_f_v     = v_vis[in_bounds]

# Pixel coordinate grids (reused per object in fused pointmap)
u_grid, v_grid = np.meshgrid(np.arange(W), np.arange(H))


In [14]:
# Visualize all projected LiDAR points — colored by depth
fig, ax = plt.subplots(1, 1, figsize=(16, 8))
ax.imshow(img_rgb)
sc = ax.scatter(u_vis, v_vis, c=Z_vis, cmap='plasma_r', s=2, alpha=0.7, vmin=0, vmax=50)
plt.colorbar(sc, ax=ax, label='depth (m)')
ax.set_title(f'LiDAR projected to image — frame {FRAME_IDX}')
ax.axis('off')
plt.tight_layout()
plt.show()

# 5. Load SAM 3 masks -- overview and instance selection

Loads pre-computed SAM 3 segmentation masks for this frame.


In [15]:
# Load SAM3 masks for this frame (class: OBJ_CLASS)
mask_path = SAM3_MASK_DIR / f'{SAM3_SCENE_PREFIX}{OBJ_CLASS}.npy'
sam3_data = np.load(str(mask_path), allow_pickle=True).item()

frame      = sam3_data[FRAME_IDX]
obj_ids    = frame['out_obj_ids']
probs      = np.asarray(frame['out_probs'], dtype=np.float32)
raw_masks  = frame['out_binary_masks']
boxes_xywh = np.asarray(frame['out_boxes_xywh'], dtype=np.float32)

# Collect all valid detections
detections = []
for i, (oid, prob) in enumerate(zip(obj_ids, probs)):
    if float(prob) < MIN_PROB:
        continue
    bm = np.asarray(raw_masks[i], dtype=bool)
    if bm.sum() < MIN_MASK_PX:
        continue
    bx = boxes_xywh[i]
    detections.append({
        'obj_id': int(oid), 'prob': float(prob),
        'binary_mask': bm,
        'box_xyxy': [float(bx[0])*W, float(bx[1])*H,
                     (float(bx[0])+float(bx[2]))*W,
                     (float(bx[1])+float(bx[3]))*H],
    })

print(f'{len(detections)} {OBJ_CLASS} detection(s) above threshold in frame {FRAME_IDX}:')
for i, d in enumerate(detections):
    n_px = d['binary_mask'].sum()
    b    = d['box_xyxy']
    print(f'  [{i}] obj_id={d["obj_id"]}  prob={d["prob"]:.3f}  '
          f'mask_px={n_px}  bbox=[{b[0]:.0f},{b[1]:.0f},{b[2]:.0f},{b[3]:.0f}]')

In [16]:
# Overview: all detected objects (for the specific SAM 3 prompt) on a single image, each with index label
_COLORS = [
    (0, 230, 0),    # green
    (255, 80, 0),   # orange
    (0, 180, 255),  # blue
    (255, 0, 200),  # magenta
    (255, 230, 0),  # yellow
    (0, 255, 200),  # teal
    (180, 0, 255),  # purple
    (255, 255, 255),# white
]

overview = img_rgb.copy().astype(np.float32)
for idx, det in enumerate(detections):
    color = _COLORS[idx % len(_COLORS)]
    mask  = det['binary_mask']
    fill  = np.zeros_like(overview)
    fill[mask] = color
    overview = overview * 0.65 + fill * 0.35

overview = overview.clip(0, 255).astype(np.uint8)

fig, ax = plt.subplots(1, 1, figsize=(16, 8))
ax.imshow(overview)

for idx, det in enumerate(detections):
    color_f = tuple(c/255 for c in _COLORS[idx % len(_COLORS)])
    x1, y1, x2, y2 = det['box_xyxy']
    rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                              lw=2, edgecolor=color_f, facecolor='none')
    ax.add_patch(rect)
    ax.text(x1+4, y1+20, f'[{idx}]',
            color=color_f, fontsize=14, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.2', fc='black', alpha=0.6))
    ax.text(x1+4, y1+42, f'p={det["prob"]:.2f}',
            color=color_f, fontsize=9,
            bbox=dict(boxstyle='round,pad=0.2', fc='black', alpha=0.5))

ax.set_title(
    f'All detected {OBJ_CLASS}s — frame {FRAME_IDX}  ({len(detections)} instances)\n'
    f'→ Set OBJ_IDXS in the config cell to the index(es) you want to analyse',
    fontsize=12
)
ax.axis('off')
plt.tight_layout()
plt.show()